# 07 · RQ2 analysis

Builds the RQ2 table with the same labels and weights as notebook 04. Input: `labels.csv`, `scores_rq1.csv`, `scores_rq2.csv`. Output: `results_rq2/rq2_metrics.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/jailbreak_project"
OUT_DIR = f"{SAVE_DIR}/results_rq2"
os.makedirs(OUT_DIR, exist_ok=True)

DET = {"protectai": "ProtectAI", "pg2_22m": "PG2-22M", "pg2_86m": "PG2-86M", "piguard": "PIGuard"}
FORMS = {"orig": "Original", "b64_wrap": "Base64 + wrapper", "b64_raw": "Base64", "rot13_wrap": "ROT13 + wrapper",
         "rot13_raw": "ROT13", "piglatin": "Pig Latin", "zulu": "Zulu"}

rq1 = pd.read_csv(f"{SAVE_DIR}/scores_rq1.csv")
x = (pd.read_csv(f"{SAVE_DIR}/labels.csv", keep_default_na=False)[["uid", "orig", "final"]]
     .merge(pd.read_csv(f"{SAVE_DIR}/scores_rq2.csv"), on="uid"))
assert len(x) == 2356

y = (x.final == "yes").values                 # our label
w = np.where(x.orig == 1, 1.0, (rq1.label == 0).sum() / (x.orig == 0).sum())


def metrics(s):
    """Weighted recall and FPR (%) at threshold 0.5, and weighted AUC."""
    flag = s >= 0.5
    return (round(100 * (w * flag)[y].sum() / w[y].sum(), 1),
            round(100 * (w * flag)[~y].sum() / w[~y].sum(), 1),
            round(roc_auc_score(y, s, sample_weight=w), 3))

## 1. Results per form

Still caught: of the jailbreaks a detector catches in the original form, how many it still catches in this form (a count, not weighted). Read it together with FPR, since a detector that flags almost every prompt also keeps almost all of them.

In [ ]:
rows = []
for d, name in DET.items():
    caught = y & (x[f"orig__{d}"].values >= 0.5)          # jailbreaks caught in the original form
    for f, form in FORMS.items():
        s = x[f"{f}__{d}"].values
        recall, fpr, auc = metrics(s)
        still = "–" if f == "orig" else f"{(caught & (s >= 0.5)).sum()} / {caught.sum()}"
        rows.append({"Detector": name, "Form": form, "Recall (%)": recall, "FPR (%)": fpr, "AUC": auc,
                     "Still caught": still})
t = pd.DataFrame(rows)
print(t.to_string(index=False))
t.to_csv(f"{OUT_DIR}/rq2_metrics.csv", index=False)

## 2. Check: rescored originals

The original prompts rescored in notebook 06 should match their notebook-01 scores.

In [ ]:
for d, name in DET.items():
    diff = np.abs(x[f"orig__{d}"].values - rq1[d].values[x.uid])
    print(f"{name}: largest difference {diff.max():.1g}")